# Step 9: MLflow Tracing and Citations

This notebook:
1. Sets up comprehensive MLflow tracing for the multi-agent system
2. Logs every agent step with inputs, outputs, and metadata
3. Provides citation provenance (which data supported which answer)
4. Creates an evaluation dataset for agent quality assessment

**Run notebooks 00-08 first!**

## 9A. Install Packages

In [0]:
%pip install -U typing-extensions pydantic
%pip install langchain langchain-groq databricks-vectorsearch mlflow sentence-transformers faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 671.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 11.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 20.2 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.4.0
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.10/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-b5685c1f-5020-4788-884b-b57be3f16453
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
  Attempting uninstall: pydantic
    Found existing installation: pydantic 1.10.6
    Not uninstalling pydantic at /databricks/python3/lib/python3.10/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-b5685c1f-5020-4788-884b-b57be3f16453
    Can't uninstall 'pydantic'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python

In [0]:
dbutils.library.restartPython()

## 9B. Configuration

In [0]:
import os, json, time, mlflow
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from databricks.vector_search.client import VectorSearchClient

CATALOG = "hackathon_vf"
SCHEMA = "healthcare"
TABLE_PREFIX = f"{CATALOG}.{SCHEMA}"

try:
    spark.sql(f"USE CATALOG {CATALOG}")
    spark.sql(f"USE SCHEMA {SCHEMA}")
except Exception:
    CATALOG = "hive_metastore"
    SCHEMA = "hackathon"
    TABLE_PREFIX = f"{CATALOG}.{SCHEMA}"
    spark.sql(f"USE {SCHEMA}")

ENRICHED_TABLE = f"{TABLE_PREFIX}.facilities_enriched"
DESERT_TABLE = f"{TABLE_PREFIX}.regional_analysis"
VS_ENDPOINT = "vf_facility_search"
VS_INDEX_NAME = f"{TABLE_PREFIX}.facilities_vs_index"
TRACE_TABLE = f"{TABLE_PREFIX}.agent_traces"

try:
    GROQ_API_KEY = dbutils.secrets.get(scope="hackathon", key="GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "YOUR_GROQ_API_KEY_HERE")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY, temperature=0, max_tokens=2048)

experiment_name = f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/vf_healthcare_agent"
mlflow.set_experiment(experiment_name)
print(f"MLflow experiment: {experiment_name}")

MLflow experiment: /Users/vm8810@srmist.edu.in/vf_healthcare_agent


## 9C. Load Components

In [0]:
vsc = VectorSearchClient()
USING_VS = False
vs_index = None
try:
    vs_index = vsc.get_index(VS_ENDPOINT, VS_INDEX_NAME)
    USING_VS = True
    print("Vector Search connected")
except Exception as e:
    print(f"VS not available: {e}, trying FAISS...")
    try:
        import faiss, pickle, tempfile, numpy as np
        from sentence_transformers import SentenceTransformer
        local_dir = tempfile.mkdtemp()
        dbutils.fs.cp("/FileStore/hackathon/vector_store/facility_index.faiss", f"file:{local_dir}/facility_index.faiss")
        dbutils.fs.cp("/FileStore/hackathon/vector_store/facility_metadata.pkl", f"file:{local_dir}/facility_metadata.pkl")
        faiss_index = faiss.read_index(f"{local_dir}/facility_index.faiss")
        with open(f"{local_dir}/facility_metadata.pkl", "rb") as f:
            faiss_meta = pickle.load(f)
        embed_model = SentenceTransformer("all-MiniLM-L6-v2")
        print(f"FAISS loaded: {faiss_index.ntotal} vectors")
    except Exception as e2:
        print(f"FAISS also unavailable: {e2}")

facilities_pdf = spark.table(ENRICHED_TABLE).toPandas()
print(f"Loaded {len(facilities_pdf)} facilities")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Vector Search connected
Loaded 797 facilities


## 9D. Sub-Agent Functions (Compact)

In [0]:
CLASSIFY_PROMPT = ChatPromptTemplate.from_template(
    "Classify into ONE: STRUCTURED (counts/aggregations), SEMANTIC (service lookups), "
    "REASONING (anomalies/inference), DESERT (gaps/underserved). Question: {question}\nCategory:")

def classify_query(question):
    cat = llm.invoke(CLASSIFY_PROMPT.format(question=question)).content.strip().upper()
    valid = {"STRUCTURED", "SEMANTIC", "REASONING", "DESERT"}
    return cat if cat in valid else next((v for v in valid if v in cat), "SEMANTIC")

def sql_sub_agent(question):
    prompt = ChatPromptTemplate.from_template(
        f"Convert to Spark SQL (SELECT only, no backticks). Table: {ENRICHED_TABLE}. "
        f"Columns: pk_unique_id, name, facilityTypeId, address_stateOrRegion, numberDoctors, capacity, "
        f"specialties, procedure, equipment, num_procedures, num_equipment, num_specialties, "
        f"flag_procedures_no_doctors, flag_capacity_no_equipment, flag_clinic_claims_surgery. "
        f"Regional: {DESERT_TABLE} (desert_score, facility_count, total_doctors). "
        "Search text: LOWER(col) LIKE '%term%'. Question: {q}\nSQL:")
    sql = llm.invoke(prompt.format(q=question)).content.strip()
    if sql.startswith("```"): sql = "\n".join(l for l in sql.split("\n") if not l.strip().startswith("```"))
    sql = sql.strip().rstrip(";")
    for kw in ["DROP","DELETE","UPDATE","INSERT","ALTER","CREATE","TRUNCATE"]:
        if kw in sql.upper(): return f"BLOCKED: {kw}", sql, []
    try:
        rows = spark.sql(sql).limit(30).collect()
        cols = spark.sql(sql).columns
        text = "\n".join(" | ".join(str(r[c]) for c in cols) for r in rows)
        sids = [str(r["pk_unique_id"]) for r in rows if "pk_unique_id" in cols] if "pk_unique_id" in cols else []
        return text, sql, sids
    except Exception as e:
        return f"Error: {e}", sql, []

def rag_sub_agent(question):
    facilities, sids = [], []
    if USING_VS and vs_index:
        res = vs_index.similarity_search(query_text=question, columns=["pk_unique_id","name","search_text","address_stateOrRegion","facilityTypeId","specialties","numberDoctors"], num_results=10)
        cols = [c["name"] for c in res["manifest"]["columns"]]
        for row in res["result"]["data_array"]:
            f = dict(zip(cols, row)); facilities.append(f); sids.append(f.get("pk_unique_id",""))
    if not facilities: return "No semantic search results.", [], []
    ctx = "\n".join(f"[{i+1}] {f.get('name','?')} ({f.get('facilityTypeId','?')}) {f.get('address_stateOrRegion','?')} Specs:{f.get('specialties','[]')}" for i,f in enumerate(facilities))
    ans = llm.invoke(ChatPromptTemplate.from_template("Healthcare agent. Answer from data. Cite names.\nDATA:\n{c}\nQ: {q}\nA:").format(c=ctx,q=question)).content
    return ans, sids, facilities

def reasoning_sub_agent(question):
    rel = facilities_pdf.sort_values("source_count", ascending=False).head(15)
    txt = "\n".join(f"- {r.get('name','?')} [{r.get('facilityTypeId','?')}] {r.get('address_stateOrRegion','?')} Docs:{r.get('numberDoctors','N/A')} Procs:{r.get('num_procedures',0)} Equip:{r.get('num_equipment',0)}" for _,r in rel.iterrows())
    ans = llm.invoke(ChatPromptTemplate.from_template("Medical reasoning agent.\nDATA:\n{d}\nQ: {q}\nA:").format(d=txt,q=question)).content
    return ans, rel["pk_unique_id"].dropna().tolist()

def desert_sub_agent(question):
    try:
        rows = spark.table(DESERT_TABLE).orderBy("desert_score", ascending=False).collect()
        data = "\n".join(f"{r['address_stateOrRegion']}: Fac={r['facility_count']}, Docs={r['total_doctors']}, Surgery={'Y' if r['has_surgery']>0 else 'N'}, Score={r['desert_score']}" for r in rows)
    except: data = "N/A"
    return llm.invoke(ChatPromptTemplate.from_template("Desert analyst.\nDATA:\n{d}\nQ: {q}\nA:").format(d=data,q=question)).content, []

## 9E. Fully Traced Agent Query

In [0]:
def traced_agent_query(question: str) -> dict:
    """Full multi-agent pipeline with comprehensive MLflow tracing."""
    with mlflow.start_run(run_name=f"query_{datetime.now().strftime('%H%M%S')}"):
        trace = {"question": question, "timestamp": datetime.now().isoformat(), "steps": []}
        mlflow.log_param("question", question[:250])

        # Step 1: Classify
        t0 = time.time()
        category = classify_query(question)
        trace["steps"].append({"step": "classify", "output": category, "ms": round((time.time()-t0)*1000)})
        mlflow.log_param("intent_category", category)
        print(f"[1] Intent: {category}")

        # Step 2: Route & Execute
        t1 = time.time()
        source_ids = []
        if category == "STRUCTURED":
            raw, sql, source_ids = sql_sub_agent(question)
            mlflow.log_text(sql, "generated_sql.sql"); agent = "sql"
        elif category == "SEMANTIC":
            raw, source_ids, _ = rag_sub_agent(question); agent = "rag"
        elif category == "REASONING":
            raw, source_ids = reasoning_sub_agent(question); agent = "reasoning"
        else:
            raw, source_ids = desert_sub_agent(question); agent = "desert"

        trace["steps"].append({"step": agent, "sources": len(source_ids), "ms": round((time.time()-t1)*1000)})
        mlflow.log_param("agent_used", agent)
        print(f"[2] Agent: {agent} ({len(source_ids)} sources)")

        # Step 3: Synthesize (SQL only)
        if category == "STRUCTURED" and not raw.startswith(("BLOCKED","Error")):
            final = llm.invoke(ChatPromptTemplate.from_template("Answer naturally.\nQ: {q}\nData:\n{d}\nA:").format(q=question,d=raw[:3000])).content
        else:
            final = raw

        # Step 4: Citations
        citations = []
        for sid in source_ids[:10]:
            if sid:
                match = facilities_pdf[facilities_pdf["pk_unique_id"]==str(sid)]
                if len(match)>0:
                    r = match.iloc[0]
                    citations.append({"id": str(sid), "name": str(r.get("name","")), "region": str(r.get("address_stateOrRegion",""))})

        total = time.time()-t0
        mlflow.log_metric("total_time_ms", round(total*1000))
        mlflow.log_metric("citation_count", len(citations))
        mlflow.log_text(final, "final_answer.txt")
        if citations: mlflow.log_text(json.dumps(citations, indent=2), "citations.json")
        mlflow.log_text(json.dumps(trace, indent=2, default=str), "trace.json")
        print(f"[3] Done: {total:.2f}s | {len(citations)} citations")

        return {"question": question, "category": category, "agent": agent,
                "answer": final, "citations": citations, "trace": trace, "total_time": round(total,2)}

## 9F. Test Traced Queries

In [0]:
print("=" * 70)
print("TRACED QUERY 1: Structured")
print("=" * 70)
r1 = traced_agent_query("How many hospitals have cardiology?")
print(f"\n{r1['answer']}\nCitations: {json.dumps(r1['citations'], indent=2)}")

TRACED QUERY 1: Structured
[1] Intent: STRUCTURED
[2] Agent: sql (0 sources)
[3] Done: 2.58s | 0 citations

Unfortunately, I don't have the most up-to-date information on the exact number of hospitals with cardiology departments. However, I can tell you that cardiology is a common specialty found in many hospitals, especially larger ones. If you're looking for a specific hospital or location, I can try to help you find that information. Alternatively, you can also check with your local healthcare provider or search online for hospitals in your area that offer cardiology services. Would you like me to try and help you with that?
Citations: []


In [0]:
print("=" * 70)
print("TRACED QUERY 2: Semantic")
print("=" * 70)
r2 = traced_agent_query("What services does Korle Bu Teaching Hospital offer?")
print(f"\n{r2['answer']}")

TRACED QUERY 2: Semantic
[1] Intent: SEMANTIC
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[2] Agent: rag (10 sources)
[3] Done: 1.37s | 0 citations

Korle Bu Teaching Hospital, as cited in [1], offers the following services: 
1. Cardiac Surgery 
2. Cardiology 
3. Family Medicine 
4. Family Planning And Complex Contraception 
5. General Surgery 
6. Gynecologic Oncology 
7. Gynecology And Obstetrics 
8. Internal Medicine 
9. Medical Oncology 
10. Nuclear Medicine And Molecular Imaging 
11. Pathology 
12. Pediatric Cardiology 
13. Pediatrics 
14. Radiology 
15. Reproductive Endocrinology And Infertility 
16. Vascular Surgery. 

These services are provided by Korle Bu Teaching Hospital, according to the data.


In [0]:
print("=" * 70)
print("TRACED QUERY 3: Reasoning")
print("=" * 70)
r3 = traced_agent_query("Which facilities have suspicious mismatches between procedures and equipment?")
print(f"\n{r3['answer']}")

TRACED QUERY 3: Reasoning
[1] Intent: REASONING
[2] Agent: reasoning (15 sources)
[3] Done: 1.85s | 0 citations

To identify facilities with suspicious mismatches between procedures and equipment, we need to look for instances where the number of procedures is significantly higher than the number of equipment or vice versa, considering that equipment is typically necessary to perform procedures. Given the data, here are some observations:

1. **Korle Bu Teaching Hospital**: 16 procedures with only 5 equipment. This mismatch could be considered suspicious as the number of procedures seems high compared to the available equipment.
2. **FirstCare Health Services**: 17 procedures with only 4 equipment. Similar to Korle Bu, the high number of procedures compared to the low number of equipment raises questions.
3. **Mother-Love Hospital**: 12 procedures with 3 equipment. This is another instance where the number of procedures seems to outweigh the available equipment significantly.
4. **MSI 

In [0]:
print("=" * 70)
print("TRACED QUERY 4: Desert")
print("=" * 70)
r4 = traced_agent_query("Where should the Virtue Foundation prioritize sending doctors?")
print(f"\n{r4['answer']}")

TRACED QUERY 4: Desert
[1] Intent: REASONING
[2] Agent: reasoning (15 sources)
[3] Done: 3.20s | 0 citations

To determine where the Virtue Foundation should prioritize sending doctors, we need to analyze the provided data, focusing on the number of doctors (Docs), procedures (Procs), and equipment (Equip) available at each facility. Since the number of doctors is not specified (nan) for most facilities, we'll have to rely on the available data for procedures and equipment as indirect indicators of need or capacity.

1. **Korle Bu Teaching Hospital**: 16 procedures, 5 equipment pieces. This hospital has a moderate number of procedures and some equipment, suggesting it might have a decent setup but could still benefit from additional medical staff.

2. **The Bank Hospital**: 3 procedures, 9 equipment pieces. Despite having fewer procedures, it has a significant amount of equipment, which might indicate a need for more staff to utilize the equipment effectively.

3. **Komfo Anokye Teachi

## 9G. Save Trace Log to Delta

In [0]:
trace_records = []
for result in [r1, r2, r3, r4]:
    trace_records.append({
        "question": result["question"], "category": result["category"],
        "agent_used": result["agent"], "answer_length": len(result["answer"]),
        "citation_count": len(result["citations"]), "total_time_seconds": float(result["total_time"]),
        "timestamp": datetime.now().isoformat(), "answer_preview": result["answer"][:500]
    })

schema = StructType([
    StructField("question", StringType()), StructField("category", StringType()),
    StructField("agent_used", StringType()), StructField("answer_length", IntegerType()),
    StructField("citation_count", IntegerType()), StructField("total_time_seconds", FloatType()),
    StructField("timestamp", StringType()), StructField("answer_preview", StringType()),
])

spark.createDataFrame(trace_records, schema=schema).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(TRACE_TABLE)
print(f"Traces saved to {TRACE_TABLE}")
display(spark.table(TRACE_TABLE).orderBy("timestamp", ascending=False))

Traces saved to hackathon_vf.healthcare.agent_traces


question,category,agent_used,answer_length,citation_count,total_time_seconds,timestamp,answer_preview
Where should the Virtue Foundation prioritize sending doctors?,REASONING,reasoning,3150,0,3.2,2026-04-02T09:10:53.598682,"To determine where the Virtue Foundation should prioritize sending doctors, we need to analyze the provided data, focusing on the number of doctors (Docs), procedures (Procs), and equipment (Equip) available at each facility. Since the number of doctors is not specified (nan) for most facilities, we'll have to rely on the available data for procedures and equipment as indirect indicators of need or capacity. 1. **Korle Bu Teaching Hospital**: 16 procedures, 5 equipment pieces. This hospital has"
Which facilities have suspicious mismatches between procedures and equipment?,REASONING,reasoning,1545,0,1.85,2026-04-02T09:10:53.598674,"To identify facilities with suspicious mismatches between procedures and equipment, we need to look for instances where the number of procedures is significantly higher than the number of equipment or vice versa, considering that equipment is typically necessary to perform procedures. Given the data, here are some observations: 1. **Korle Bu Teaching Hospital**: 16 procedures with only 5 equipment. This mismatch could be considered suspicious as the number of procedures seems high compared to t"
What services does Korle Bu Teaching Hospital offer?,SEMANTIC,rag,565,0,1.37,2026-04-02T09:10:53.598665,"Korle Bu Teaching Hospital, as cited in [1], offers the following services: 1. Cardiac Surgery 2. Cardiology 3. Family Medicine 4. Family Planning And Complex Contraception 5. General Surgery 6. Gynecologic Oncology 7. Gynecology And Obstetrics 8. Internal Medicine 9. Medical Oncology 10. Nuclear Medicine And Molecular Imaging 11. Pathology 12. Pediatric Cardiology 13. Pediatrics 14. Radiology 15. Reproductive Endocrinology And Infertility 16. Vascular Surgery. These services a"
How many hospitals have cardiology?,STRUCTURED,sql,527,0,2.58,2026-04-02T09:10:53.598644,"Unfortunately, I don't have the most up-to-date information on the exact number of hospitals with cardiology departments. However, I can tell you that cardiology is a common specialty found in many hospitals, especially larger ones. If you're looking for a specific hospital or location, I can try to help you find that information. Alternatively, you can also check with your local healthcare provider or search online for hospitals in your area that offer cardiology services. Would you like me to"


## 9H. Must-Have Evaluation Dataset

In [0]:
eval_questions = [
    {"id": "1.1", "question": "How many hospitals have cardiology?", "type": "STRUCTURED"},
    {"id": "1.3", "question": "What services does Korle Bu Teaching Hospital offer?", "type": "SEMANTIC"},
    {"id": "1.5", "question": "Which region has the most hospital-type facilities?", "type": "STRUCTURED"},
    {"id": "2.3", "question": "Where are geographic cold spots for critical procedures?", "type": "DESERT"},
    {"id": "4.4", "question": "Which facilities claim unrealistic procedures for their size?", "type": "REASONING"},
    {"id": "4.9", "question": "Where do we see things that shouldn't move together?", "type": "REASONING"},
    {"id": "6.1", "question": "Where is the surgical workforce practicing in Ghana?", "type": "REASONING"},
    {"id": "7.5", "question": "Which procedures depend on only 1-2 facilities per region?", "type": "STRUCTURED"},
    {"id": "7.6", "question": "Where is oversupply vs scarcity of procedures?", "type": "REASONING"},
    {"id": "8.3", "question": "Where are gaps where no NGOs work despite need?", "type": "DESERT"},
]
print(f"Evaluation: {len(eval_questions)} Must-Have questions")
for eq in eval_questions:
    print(f"  [{eq['id']}] {eq['type']:12s} | {eq['question']}")

Evaluation: 10 Must-Have questions
  [1.1] STRUCTURED   | How many hospitals have cardiology?
  [1.3] SEMANTIC     | What services does Korle Bu Teaching Hospital offer?
  [1.5] STRUCTURED   | Which region has the most hospital-type facilities?
  [2.3] DESERT       | Where are geographic cold spots for critical procedures?
  [4.4] REASONING    | Which facilities claim unrealistic procedures for their size?
  [4.9] REASONING    | Where do we see things that shouldn't move together?
  [6.1] REASONING    | Where is the surgical workforce practicing in Ghana?
  [7.5] STRUCTURED   | Which procedures depend on only 1-2 facilities per region?
  [7.6] REASONING    | Where is oversupply vs scarcity of procedures?
  [8.3] DESERT       | Where are gaps where no NGOs work despite need?


In [0]:
print("=" * 60)
print("STEP 9 COMPLETE: MLflow Tracing & Citations")
print("=" * 60)
print(f"""
Built: Step-level MLflow tracing, citation provenance, trace Delta table,
       evaluation dataset with {len(eval_questions)} Must-Have questions.

MLflow artifacts per query: trace.json, citations.json, final_answer.txt
Table: {TRACE_TABLE}
Function: traced_agent_query(question)

Next: Run notebook 10_final_testing
""")

STEP 9 COMPLETE: MLflow Tracing & Citations

Built: Step-level MLflow tracing, citation provenance, trace Delta table,
       evaluation dataset with 10 Must-Have questions.

MLflow artifacts per query: trace.json, citations.json, final_answer.txt
Table: hackathon_vf.healthcare.agent_traces
Function: traced_agent_query(question)

Next: Run notebook 10_final_testing

